# Day 66: Training Deep Networks - Optimizers, Epochs, Batch Size

## Optimizers
The optimizer is the algorithm that decides how much to change each weight after computing the gradient of the loss - it turns "which direction reduces error" into an actual update rule. Plain gradient descent (SGD) takes a fixed-size step in the gradient direction every time, which can be slow and easy to get stuck on. Adam combines two ideas - momentum (remembering the direction you were moving in) and per-parameter adaptive learning rates - so it usually converges faster and needs less tuning, which is why it's the default choice for most deep learning problems. The code below builds a small binary classifier that we'll reuse to compare optimizers: two Dense layers, a ReLU hidden layer, and a sigmoid output for a 0/1 prediction.</cell id="cell-1">


In [ ]:
# pip install tensorflow
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

np.random.seed(0)  # fixed seed so the random data is reproducible across runs
X = np.random.randn(200, 4)  # 200 samples, 4 random features
y = (X.sum(axis=1) > 0).astype(int)  # simple synthetic label: 1 if the features sum to a positive value, else 0

def build_model():
    # a fresh, untrained model each time so different optimizers start from the same conditions
    return keras.Sequential([
        layers.Dense(16, activation='relu', input_shape=(4,)),  # hidden layer learns nonlinear feature combinations
        layers.Dense(1, activation='sigmoid')  # single output squashed to [0,1] for binary classification
    ])

## Comparing Optimizers
Not every optimizer behaves the same on every dataset, so it's a good habit to try a few before committing to one for a real project. Here we train the exact same model architecture on the exact same data three times, changing only the optimizer string passed to `compile` - `sgd`, `adam`, and `rmsprop`. Each run reports the final training accuracy after 20 epochs, so you can directly compare how quickly and how well each optimizer fits the data. In practice `adam` tends to win on small/medium problems like this because of its adaptive learning rate, but seeing `sgd` lag behind helps build intuition for why it's rarely used untuned.</cell id="cell-3">


In [ ]:
for opt in ['sgd', 'adam', 'rmsprop']:
    model = build_model()  # new, untrained model for a fair comparison
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])  # binary_crossentropy is the standard loss for 2-class problems
    history = model.fit(X, y, epochs=20, verbose=0)  # verbose=0 keeps the output clean; history stores per-epoch metrics
    print(opt, '-> final accuracy:', round(history.history['accuracy'][-1], 3))  # accuracy from the last epoch only

## Epochs & Batch Size
An epoch is one complete pass through the entire training set, and batch size is how many samples the model looks at before updating its weights once. Smaller batches give noisier but more frequent updates (which can help escape bad local minima), while larger batches give smoother, more stable but slower updates - batch_size=16 here is a common middle ground for small datasets. `validation_split=0.2` holds out 20% of the data (not used for training) purely to check how well the model generalizes to unseen data after each epoch. Comparing the final train accuracy to the final validation accuracy is the simplest way to spot overfitting: if train accuracy is much higher than validation accuracy, the model has memorized the training data rather than learning general patterns.</cell id="cell-5">


In [ ]:
model = build_model()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# batch_size=16 -> weight update every 16 samples; validation_split=0.2 holds out 20% of X/y for evaluation only
history = model.fit(X, y, epochs=30, batch_size=16, validation_split=0.2, verbose=0)
print('Final train accuracy:', history.history['accuracy'][-1])
print('Final val accuracy:', history.history['val_accuracy'][-1])  # compare to train accuracy to check for overfitting

## Plotting Training Curves
Numbers printed at the end of training only tell you the final state, but plotting accuracy over every epoch shows the whole learning story. If both the train and validation curves rise together and stay close, the model is learning general patterns well. If the train curve keeps climbing while the validation curve flattens or starts dropping, that gap is the visual signature of overfitting - the model is fitting noise specific to the training set. Watching this plot is one of the fastest, most common diagnostic habits in deep learning, since it tells you whether to add regularization, stop training earlier, or gather more data.</cell id="cell-7">


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='Train')  # accuracy on the training data per epoch
plt.plot(history.history['val_accuracy'], label='Validation')  # accuracy on the held-out validation data per epoch
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training Curves')
plt.show()